# Conversion Rate vs. Contact Metrics by General Manager

This notebook analyzes the relationship between conversion rates and various contact metrics, grouped by General Manager.

**Filtering criteria:**
- Excludes Licensee locations
- General Managers with 200+ reservations
- Excludes GMs with >95% no contact rate (data anomalies)

*Note: Licensee locations operate on a different model (franchise dealerships with ~98% no contact rate). They would be automatically excluded by the >95% no contact filter even without explicit exclusion.*

**Charts included:**
1. Conversion Rate vs. % Under 30 Minutes Contact
2. Conversion Rate vs. % Counter Contact
3. Conversion Rate vs. % No Contact
4. Conversion Rate vs. % Under 1 Hour Contact

**Bubble color = HTZ Region**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load data
file_path = '../data/raw/Conversion Data Nov-Dec 2025 (1).xlsx'
df = pd.read_excel(file_path, engine='openpyxl')

# Clean column names
df.columns = df.columns.str.strip()

# Exclude Licensee locations
licensee_count = (df['HTZREGION'] == 'LICENSEE').sum()
df = df[df['HTZREGION'] != 'LICENSEE']
print(f"Excluded {licensee_count:,} Licensee records")

print(f"Total records (after exclusion): {len(df):,}")
print(f"\nUnique General Managers: {df['GENERAL_MGR'].nunique()}")
print(f"Unique HTZ Regions: {df['HTZREGION'].nunique()}")
print(f"\nHTZ Regions: {df['HTZREGION'].unique()}")

In [ ]:
# Calculate metrics by General Manager
# Also get the most common region for each manager
manager_stats = df.groupby('GENERAL_MGR').agg(
    total_reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum'),
    under_30min_count=('CONTACT RANGE', lambda x: (x == '(a)<30min').sum()),
    under_1hr_count=('CONTACT RANGE', lambda x: (x.isin(['(a)<30min', '(b)31min - 1hr'])).sum()),
    no_contact_count=('CONTACT RANGE', lambda x: (x == 'NO CONTACT').sum()),
    counter_count=('CONTACT_GROUP', lambda x: (x == 'COUNTER').sum()),
    total_contacts=('CONTACT RANGE', 'count'),
    region=('HTZREGION', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown')
).reset_index()

# Calculate rates
manager_stats['conversion_rate'] = (manager_stats['conversions'] / manager_stats['total_reservations'] * 100)
manager_stats['pct_under_30min'] = (manager_stats['under_30min_count'] / manager_stats['total_contacts'] * 100)
manager_stats['pct_under_1hr'] = (manager_stats['under_1hr_count'] / manager_stats['total_contacts'] * 100)
manager_stats['pct_no_contact'] = (manager_stats['no_contact_count'] / manager_stats['total_contacts'] * 100)
manager_stats['pct_counter'] = (manager_stats['counter_count'] / manager_stats['total_contacts'] * 100)

# Filter to managers with meaningful volume (at least 200 reservations - matching confounding analysis)
manager_stats_filtered = manager_stats[manager_stats['total_reservations'] >= 200].copy()

print(f"Total General Managers: {len(manager_stats)}")
print(f"General Managers with 200+ reservations: {len(manager_stats_filtered)}")

manager_stats_filtered.sort_values('total_reservations', ascending=False).head(15)

In [ ]:
# Create color mapping for regions
regions = manager_stats_filtered['region'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(regions)))
region_color_map = dict(zip(regions, colors))

print("Region Color Mapping:")
for region in regions:
    count = len(manager_stats_filtered[manager_stats_filtered['region'] == region])
    print(f"  {region}: {count} managers")

In [ ]:
# Calculate correlations for all metrics (excluding GMs with >95% no contact)
manager_stats_plot = manager_stats_filtered[manager_stats_filtered['pct_no_contact'] < 95].copy()

correlations = {
    '% Under 30min': manager_stats_plot['pct_under_30min'].corr(manager_stats_plot['conversion_rate']),
    '% Under 1hr': manager_stats_plot['pct_under_1hr'].corr(manager_stats_plot['conversion_rate']),
    '% Counter': manager_stats_plot['pct_counter'].corr(manager_stats_plot['conversion_rate']),
    '% No Contact': manager_stats_plot['pct_no_contact'].corr(manager_stats_plot['conversion_rate'])
}

print(f"Correlations with Conversion Rate (General Manager Level)")
print(f"Filter: 200+ reservations, <95% no contact rate")
print(f"N = {len(manager_stats_plot)} General Managers\n")
for metric, corr in correlations.items():
    print(f"  {metric}: {corr:+.3f}")

## Chart 1: Conversion Rate vs. % Under 30 Minutes Contact

In [ ]:
# Chart 1: Conversion Rate vs. % Under 30 Minutes
fig, ax = plt.subplots(figsize=(12, 8))

# Use filtered data (excludes >95% no contact)
sizes = manager_stats_plot['total_reservations'] / manager_stats_plot['total_reservations'].max() * 200 + 20
colors = [region_color_map[r] for r in manager_stats_plot['region']]

scatter = ax.scatter(
    manager_stats_plot['pct_under_30min'],
    manager_stats_plot['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors,
    edgecolors='black',
    linewidth=0.5
)

# Create legend for regions
regions_in_plot = manager_stats_plot['region'].unique()
legend_elements = [plt.scatter([], [], c=[region_color_map[r]], s=100, label=r, edgecolors='black', linewidth=0.5) 
                   for r in regions_in_plot]
ax.legend(handles=legend_elements, title='HTZ Region', loc='lower right', fontsize=8)

ax.set_xlabel('% of Contacts Under 30 Minutes', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Quick Contact Rate by General Manager\n(bubble size = reservation volume, color = HTZ Region)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_conversion = manager_stats_plot['conversion_rate'].mean()
avg_under30 = manager_stats_plot['pct_under_30min'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_under30, color='blue', linestyle='--', alpha=0.5, label=f'Avg <30min: {avg_under30:.1f}%')

corr = manager_stats_plot['pct_under_30min'].corr(manager_stats_plot['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

## Chart 2: Conversion Rate vs. % Counter Contact

In [ ]:
# Chart 2: Conversion Rate vs. % Counter
fig, ax = plt.subplots(figsize=(12, 8))

# Use filtered data (excludes >95% no contact)
sizes = manager_stats_plot['total_reservations'] / manager_stats_plot['total_reservations'].max() * 200 + 20
colors = [region_color_map[r] for r in manager_stats_plot['region']]

scatter = ax.scatter(
    manager_stats_plot['pct_counter'],
    manager_stats_plot['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors,
    edgecolors='black',
    linewidth=0.5
)

# Create legend for regions
regions_in_plot = manager_stats_plot['region'].unique()
legend_elements = [plt.scatter([], [], c=[region_color_map[r]], s=100, label=r, edgecolors='black', linewidth=0.5) 
                   for r in regions_in_plot]
ax.legend(handles=legend_elements, title='HTZ Region', loc='lower right', fontsize=8)

ax.set_xlabel('% of Contacts via Counter', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Counter Contact Rate by General Manager\n(bubble size = reservation volume, color = HTZ Region)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_counter = manager_stats_plot['pct_counter'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5)
ax.axvline(x=avg_counter, color='blue', linestyle='--', alpha=0.5)

corr = manager_stats_plot['pct_counter'].corr(manager_stats_plot['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

## Chart 3: Conversion Rate vs. % No Contact

In [ ]:
# Chart 3: Conversion Rate vs. % No Contact (excluding extreme outliers >95%)
fig, ax = plt.subplots(figsize=(12, 8))

# Filter to managers with <95% no contact (matching confounding analysis)
no_contact_filtered = manager_stats_filtered[manager_stats_filtered['pct_no_contact'] < 95].copy()

sizes = no_contact_filtered['total_reservations'] / no_contact_filtered['total_reservations'].max() * 200 + 20
colors = [region_color_map[r] for r in no_contact_filtered['region']]

scatter = ax.scatter(
    no_contact_filtered['pct_no_contact'],
    no_contact_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors,
    edgecolors='black',
    linewidth=0.5
)

# Create legend for regions
regions_in_chart = no_contact_filtered['region'].unique()
legend_elements = [plt.scatter([], [], c=[region_color_map[r]], s=100, label=r, edgecolors='black', linewidth=0.5) 
                   for r in regions_in_chart]
ax.legend(handles=legend_elements, title='HTZ Region', loc='upper right', fontsize=8)

ax.set_xlabel('% of No Contact', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. No Contact Rate by General Manager\n(bubble size = reservation volume, color = HTZ Region, excludes >95% no contact)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_no_contact = no_contact_filtered['pct_no_contact'].mean()
avg_conv_no_contact = no_contact_filtered['conversion_rate'].mean()
ax.axhline(y=avg_conv_no_contact, color='red', linestyle='--', alpha=0.5)
ax.axvline(x=avg_no_contact, color='blue', linestyle='--', alpha=0.5)

corr = no_contact_filtered['pct_no_contact'].corr(no_contact_filtered['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

print(f"Showing {len(no_contact_filtered)} General Managers with <95% no contact (excluded {len(manager_stats_filtered) - len(no_contact_filtered)} managers with extreme no-contact rates)")

## Chart 4: Conversion Rate vs. % Under 1 Hour Contact

In [ ]:
# Chart 4: Conversion Rate vs. % Under 1 Hour
fig, ax = plt.subplots(figsize=(12, 8))

# Use filtered data (excludes >95% no contact)
sizes = manager_stats_plot['total_reservations'] / manager_stats_plot['total_reservations'].max() * 200 + 20
colors = [region_color_map[r] for r in manager_stats_plot['region']]

scatter = ax.scatter(
    manager_stats_plot['pct_under_1hr'],
    manager_stats_plot['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors,
    edgecolors='black',
    linewidth=0.5
)

# Create legend for regions
regions_in_plot = manager_stats_plot['region'].unique()
legend_elements = [plt.scatter([], [], c=[region_color_map[r]], s=100, label=r, edgecolors='black', linewidth=0.5) 
                   for r in regions_in_plot]
ax.legend(handles=legend_elements, title='HTZ Region', loc='lower right', fontsize=8)

ax.set_xlabel('% of Contacts Under 1 Hour', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Under 1 Hour Contact Rate by General Manager\n(bubble size = reservation volume, color = HTZ Region)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_under_1hr = manager_stats_plot['pct_under_1hr'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5)
ax.axvline(x=avg_under_1hr, color='blue', linestyle='--', alpha=0.5)

corr = manager_stats_plot['pct_under_1hr'].corr(manager_stats_plot['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

## Summary: Top 15 General Managers by Volume

In [ ]:
# Top 15 General Managers by volume with all metrics (from filtered data)
print("Top 15 General Managers by Volume (200+ reservations, <95% no contact):")
manager_stats_plot.nlargest(15, 'total_reservations')[
    ['GENERAL_MGR', 'region', 'total_reservations', 'conversion_rate', 
     'pct_under_30min', 'pct_under_1hr', 'pct_counter', 'pct_no_contact']
].round(1)